In [19]:
import sys
import pandas as pd
from pathlib import Path

# This moves from notebooks/ up one level to project root
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


Using project root: C:\Users\nathan.taylor\Jupyter Notebooks\outlier_pipeline


In [20]:
from src.runner import run_all_jobs

CONFIG_PATH = PROJECT_ROOT / "configs" / "jobs_smart_offer.yml"

results = run_all_jobs(str(CONFIG_PATH))


2026-02-12 13:31:14,727 [INFO] src.runner - Starting job: ATT_MOB_smart_offer
2026-02-12 13:31:14,807 [INFO] pyhive.presto - WITH
DateSelector AS (
    SELECT *
    FROM
        ( VALUES (
             DATE ('2026-01-30'), DATE ('2026-02-13'),
                 '2026-01-30', '2026-02-13'
                 ))
        AS t ("StartDate","EndDate","StartDateStr","EndDateStr")
),

-- WITH
-- DateSelector As (
--     SELECT *
--     FROM
--         ( VALUES (
--             DATE('2025-09-16'), CURRENT_DATE,
--                  '2025-09-16'
--                  ))
--         as t ("StartDate","EndDate","StartDateStr")
-- ),

SmartOfferExWo AS(
    SELECT
        COALESCE(
            element_at(ex.edp_raw_data_map, 'Identities_ReservationSid'),
            element_at(ex.edp_raw_data_map, 'Identities_SessionId')
            ) as reservation_id,
        1 AS smartoffermodelmatch,
        MAX(CASE
            WHEN ELEMENT_AT(ex.edp_raw_data_map,
                            'ExtraData_behavior') = '

In [21]:
print("\n===== JOB SUMMARY =====\n")

summary_df = pd.DataFrame([
    {
        "job_name": job_name,
        "rows_raw": meta.get("rows_raw"),
        "rows_valid": meta.get("rows_valid"),
        "rows_eligible": meta.get("rows_eligible"),
        "rows_out": meta.get("rows_out"),
        "status": meta.get("status"),
        "output_file": meta.get("out_path"),
    }
    for job_name, meta in results.items()
])

summary_df



===== JOB SUMMARY =====



,job_name,rows_raw,rows_valid,rows_eligible,rows_out,status,output_file
0,ATT_MOB_smart_offer,704,554,554,119,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...


In [22]:
job_key = "ATT_MOB_smart_offer"   # or whatever key appears in results
df_job = results[job_key]["df"]

# show whether the flag column exists
print("flag col present:", "cancellation_rate__is_outlier" in df_job.columns)

# counts of the flag (should be only True rows if you filtered)
if "cancellation_rate__is_outlier" in df_job.columns:
    print(df_job["cancellation_rate__is_outlier"].value_counts(dropna=False))


flag col present: False


In [23]:
import inspect
import src.runner

print("Has output shaping logic?:",
      "filter_to_comparison_outliers" in inspect.getsource(src.runner.run_job))

Has output shaping logic?: True


In [24]:
meta = results["ATT_MOB_smart_offer"]
print("rows_out:", meta.get("rows_out"))

rows_out: 119


In [25]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG_PATH).read_text(encoding="utf-8-sig"))
job = next(j for j in cfg["jobs"] if j["name"] == "ATT_MOB_smart_offer")

print("Top-level output:", job.get("output"))
print("Top-level keys:", sorted(job.keys()))

Top-level output: None
Top-level keys: ['coaching_override', 'comparisons', 'eligibility', 'inputs', 'name', 'query', 'reference_link', 'selection', 'validity']


In [26]:
print(results.keys())

dict_keys(['ATT_MOB_smart_offer'])
